<a href="https://colab.research.google.com/github/achuntya/ML-project/blob/main/3rd_strategy(ext_1)_KPIsFramework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git

  Cloning https://github.com/rongardF/tvdatafeed.git to /tmp/pip-req-build-76v80zv6
  Running command git clone --filter=blob:none --quiet https://github.com/rongardF/tvdatafeed.git /tmp/pip-req-build-76v80zv6
  Resolved https://github.com/rongardF/tvdatafeed.git to commit e6f6aaa7de439ac6e454d9b26d2760ded8dc4923
  Preparing metadata (setup.py) ... done


In [ ]:
!pip install pandas_ta

In [ ]:
!pip install gspread
!pip install oauth2client


In [ ]:
import pandas as pd
import numpy as np
import math
import inspect
import pandas_ta as ta
from tvDatafeed import TvDatafeed, Interval
from datetime import datetime, timedelta
import re
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# Initialize the data feed
tv = TvDatafeed()




# Function to clean stock symbol
def clean_symbol(symbol):
    return re.sub(r'[^\w]', '_', symbol)

# Function to determine the number of bars to fetch based on the timeframe and date range
def calculate_n_bars(start_date, end_date, timeframe):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    if timeframe == Interval.in_monthly:
        n_bars = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month) + 1
    elif timeframe == Interval.in_weekly:
        n_bars = (end_date - start_date).days // 7 + 1
    elif timeframe == Interval.in_daily:
        n_bars = (end_date - start_date).days + 1
    else:
        n_bars = 500  # default value, adjust as necessary

    return n_bars


#########################################################################################################################

# Define function to fetch data and calculate indicators
def fetch_data_and_calculate_indicators(symbol):
    try:
        symbol = clean_symbol(symbol)  # Clean the symbol
        n_bars = calculate_n_bars(START_DATE, END_DATE, TIMEFRAME)
        data = tv.get_hist(symbol=symbol, exchange='NSE', interval=TIMEFRAME, n_bars=n_bars)


        # Calculate indicators by executing the code in INDICATOR_CODE


        df=pd.DataFrame(data)
        if df.empty:
            print(f"No data found for {symbol}. Skipping...")
            return None
        df['datetime'] = df.index

        # Convert 'datetime' column to datetime objects
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Filter data within the date range
        df = df[(df['datetime'] >= pd.to_datetime(START_DATE)) & (df['datetime'] <= pd.to_datetime(END_DATE))]
        if 'close' not in df.columns:
            print(f"'close' column not found in the data for {symbol}. Skipping...")
            return None
        local_scope = {'df': df, 'data': data, 'ta': ta}  # Include all necessary variables
        exec(INDICATOR_CODE, globals(), local_scope)
        # works till here

        # Update the df with the modified one from local_scope

        df = local_scope['df']




        # Ensure the original 'datetime' column is preserved

        #df['datetime'] = df_original['datetime']


        # Drop rows with NaN values generated by indicator calculations
        df.dropna(inplace=True)
        # print(df)
        return df
    except Exception as e:
        print(f"An error occurred for {symbol}: {e}")
        return None


#################################################################################################################################



def apply_strategy(df):
    buy_signals = []
    sell_signals = []

    position_open = False  # Track whether a position is currently open
    entry_price = 0


    for i in range(1, len(df)-1): # Changed to len(df)-1 to avoid out-of-bounds error
        # print(ENTRY_CONDITION, EXIT_CONDITION)
        if not position_open and ENTRY_CONDITION(df, i):
            if df['open'].iloc[i+1] < MAX_BUY_PRICE:  # Only consider trades with buy price < MAX_BUY_PRICE
                buy_signals.append((df['datetime'].iloc[i+1], df['open'].iloc[i+1]))
                position_open = True
                entry_price = df['open'].iloc[i+1]
                #print(f"Buy signal: Date: {df['datetime'].iloc[i]}, Price: {df['close'].iloc[i]}")
        elif position_open:
            # Check stop-loss condition
            if STOP_LOSS is not None and df['low'].iloc[i] <= entry_price * (1 - STOP_LOSS):
                sell_signals.append((df['datetime'].iloc[i+1], entry_price * (1 - STOP_LOSS)))
                position_open = False
                #print(f"Stop-loss triggered: Date: {df['datetime'].iloc[i]}, Price: {df['close'].iloc[i]}, Entry Price: {entry_price}")

            # Check target condition
            elif TARGET is not None and df['high'].iloc[i] >= entry_price * (1 + TARGET):
                sell_signals.append((df['datetime'].iloc[i+1], entry_price * (1 + TARGET)))
                position_open = False
                #print(f"Target hit: Date: {df['datetime'].iloc[i]}, Price: {df['close'].iloc[i]}, Entry Price: {entry_price}")

            # Check exit condition
            elif EXIT_CONDITION(df, i):
                sell_signals.append((df['datetime'].iloc[i+1], df['open'].iloc[i+1]))
                position_open = False
                #print(f"Sell signal: Date: {df['datetime'].iloc[i]}, Price: {df['close'].iloc[i]}")

    return buy_signals, sell_signals


############################################################################################################



def Stock(symbol):
    global START_DATE, END_DATE, TIMEFRAME, INDICATOR_CODE, MAX_BUY_PRICE, ENTRY_CONDITION, EXIT_CONDITION, TARGET, STOP_LOSS


    df = fetch_data_and_calculate_indicators(symbol)
    # print(df)
    if df is None:
        return None, True
    # print(df.columns)
    buy_signals, sell_signals = apply_strategy(df)

    paired_trades = zip(buy_signals, sell_signals)
    trades = []

    for buy_signal, sell_signal in paired_trades:
        buy_date, buy_price = buy_signal
        sell_date, sell_price = sell_signal
        return_pct = (sell_price - buy_price) / buy_price * 100
        holding_period = (sell_date - buy_date).days
        trades.append([symbol, buy_date, sell_date, buy_price, sell_price, return_pct, holding_period])
    print(trades)
    if trades:
        trades_df = pd.DataFrame(trades, columns=['Stock Symbol', 'Buy Date', 'Sell Date', 'Buy Price', 'Sell Price', 'Return (%)', 'Holding Period'])

        avg_return_pct = trades_df['Return (%)'].mean()
        avg_holding_period = trades_df['Holding Period'].mean()
        strike_rate = len(trades_df[trades_df['Return (%)'] > 0]) / len(trades_df) * 100
        max_profit_pct = trades_df['Return (%)'].max()
        max_loss_pct = trades_df['Return (%)'].min()
        avg_profit_pct = trades_df[trades_df['Return (%)'] > 0]['Return (%)'].mean()
        avg_loss_pct = trades_df[trades_df['Return (%)'] <= 0]['Return (%)'].mean()
        risk_reward = avg_profit_pct / abs(avg_loss_pct) if avg_loss_pct != 0 else np.nan

        print(f'Stock: {symbol}')
        print(f'Number of Trades: {len(trades)}')
        print(f'Average Holding Period (days): {avg_holding_period}')
        print(f'Strike Rate (%): {strike_rate}')
        print(f'Average Return (%): {avg_return_pct}')
        print(f'Max Profit (%): {max_profit_pct}')
        print(f'Max Loss (%): {max_loss_pct}')
        print(f'Average Return on Profitable Trade (%): {avg_profit_pct}')
        print(f'Average Loss on Losing Trade (%): {avg_loss_pct}')
        print(f'Risk to Reward Ratio: {risk_reward}')
        print(" ")

        return trades, False

    return None, False


#####################################################################################################################################
def main(output_file='trade_log.csv'):
    all_trades = []
    failed_stocks = []

    entry_condition_code = inspect.getsource(ENTRY_CONDITION).strip()
    exit_condition_code = inspect.getsource(EXIT_CONDITION).strip()
    strategy_summary = f'Timeframe: {TIMEFRAME}\n Start Date: {START_DATE}\n End Date: {END_DATE}\n Max Buy Price: {MAX_BUY_PRICE}\n Target: {TARGET}\n Stop Loss: {STOP_LOSS}\n  {entry_condition_code}\n  {exit_condition_code}\n'

    for index, row in symbols_df.iterrows():
        stock_name = row['Company Name']
        stock_symbol = row['Symbol']
        print(f'Processing {stock_name} ({stock_symbol})')

        trades, failed = Stock(stock_symbol)
        if trades:
            all_trades.extend(trades)
        if failed:
            failed_stocks.append(stock_symbol)

    trades_df = pd.DataFrame(all_trades, columns=[ 'Stock Symbol', 'Buy Date', 'Sell Date', 'Buy Price', 'Sell Price', 'Return (%)', 'Holding Period'])
    trades_df.to_csv(output_file, index=False)
    print(f'Trade log saved to {output_file}')

    # Read and print the generated trade log
    trades_df = pd.read_csv(output_file)
    print("Trades log:\n", trades_df)

    # Print failed stocks
    if failed_stocks:
        print("Failed to process the following stocks:")
        for stock in failed_stocks:
            print(stock)

    # Calculate and print overall statistics
    if not trades_df.empty:
        avg_return_pct = trades_df['Return (%)'].mean()
        avg_holding_period = trades_df['Holding Period'].mean()
        total_trades = len(trades_df)
        strike_rate = len(trades_df[trades_df['Return (%)'] > 0]) / total_trades * 100
        max_profit_pct = trades_df['Return (%)'].max()
        max_loss_pct = trades_df['Return (%)'].min()
        avg_profit_pct = trades_df[trades_df['Return (%)'] > 0]['Return (%)'].mean()
        avg_loss_pct = trades_df[trades_df['Return (%)'] <= 0]['Return (%)'].mean()
        risk_reward = avg_profit_pct / abs(avg_loss_pct) if avg_loss_pct != 0 else np.nan
        daily_returns = avg_return_pct / avg_holding_period if avg_holding_period > 0 else 0
        avg_holding_period_winning = trades_df[trades_df['Return (%)'] > 0]['Holding Period'].mean()
        avg_holding_period_losing = trades_df[trades_df['Return (%)'] <= 0]['Holding Period'].mean()
        bre_sr=100.0/(risk_reward+1)
        n_stocks=trades_df['Stock Symbol'].nunique()
        R1=100*avg_return_pct*(strike_rate-bre_sr)
        R2=100000*avg_profit_pct/avg_holding_period_winning
        R3=10000000/abs(avg_loss_pct*avg_holding_period_losing)
        R4=10000*math.log(1+total_trades/n_stocks)
        R5=math.exp(abs(avg_profit_pct/avg_loss_pct))

        performance_summary = (f'Total Trades: {total_trades}\n Average Return (%): {avg_return_pct}\n Average Holding Period (days): {avg_holding_period}\n Strike Rate (%): {strike_rate}\n Max Profit (%): {max_profit_pct}\n Max Loss (%): {max_loss_pct}\n Average Profit (%): {avg_profit_pct}\n Average Loss (%): {avg_loss_pct}\n Risk to Reward Ratio: {risk_reward}\nAverage Holding Period for Winning Trades (days): {avg_holding_period_winning}\n Average Holding Period for Losing Trades (days): {avg_holding_period_losing}\n')



        print(f'Overall Average Return (%): {avg_return_pct}')
        print(f'Overall Average Holding Period (days): {avg_holding_period}')
        print(f'Total Number of Trades: {total_trades}')
        print(f'Overall Strike Rate (%): {strike_rate}')
        print(f'Overall Max Profit (%): {max_profit_pct}')
        print(f'Overall Max Loss (%): {max_loss_pct}')
        print(f'Average Return on Profitable Trade (%): {avg_profit_pct}')
        print(f'Average Loss on Losing Trade (%): {avg_loss_pct}')
        print(f'Reward to Risk Ratio: {risk_reward}')
        print(f'Daily returns:{ daily_returns}')
        print(f'Average Holding Period for Winning Trades (days): {avg_holding_period_winning}')
        print(f'Average Holding Period for Losing Trades (days): {avg_holding_period_losing}')
        print(f'R1:{R1}')
        print(f'R2:{R2}')
        print(f'R3:{R3}')
        print(f'R4:{R4}')
        print(f'R5:{R5}')
        return  NAME,strategy_summary,performance_summary, daily_returns

    return  NAME,strategy_summary,None, None

In [ ]:
# Global variables
START_DATE = '2015-10-14'
END_DATE = '2025-1-24'
TIMEFRAME = Interval.in_weekly  # Changed to weekly timeframe
universe = 'nifty500'  # Remains the same

# Strategy
NAME = "ADX and EMA,SMA"
MAX_BUY_PRICE = 10000

ENTRY_CONDITION = lambda df, i: (
  (df['ADX_6'].iloc[i] > 25 and df['DMP_6'].iloc[i] > df['DMN_6'].iloc[i]) and
  (df['ema_5'].iloc[i] > df['sma_6'].iloc[i] and df['ema_5'].iloc[i-1] < df['sma_6'].iloc[i-1])

)

EXIT_CONDITION = lambda df, i: (
    (df['ADX_6'].iloc[i] < 35 and df['DMP_6'].iloc[i] < df['DMN_6'].iloc[i]) and
    (df['ema_5'].iloc[i] < df['sma_6'].iloc[i] and df['ema_5'].iloc[i-1] > df['sma_6'].iloc[i-1])

)

TARGET = None  # Example: 0.15 for 15% target, set to None if not used
STOP_LOSS = 0.1  # Example: 0.1 for 10% stop loss, set to None if not used

# Indicator calculation code as a string
INDICATOR_CODE= """
# Calculate EMA and SMA and RSI
df['ema_5'] = ta.ema(df['close'], length = 5)
df['sma_6'] = ta.sma(df['close'], length = 6)
df['ADX_6'] = ta.adx(df['high'], df['low'], df['close'], length=6)['ADX_6']
df['DMP_6'] = ta.adx(df['high'], df['low'], df['close'], length=6)['DMP_6']  # Positive DI (+DI)
df['DMN_6'] = ta.adx(df['high'], df['low'], df['close'], length=6)['DMN_6']
df['rsi'] = ta.rsi(df['close'],length=14)
"""



In [ ]:
# Results for large Cap,Mid Cap,Small Cap,Micro Cap
# Large Cap
Overall Average Return (%): 9.175430720826046
Overall Average Holding Period (days): 145.7054794520548
Total Number of Trades: 292
Overall Strike Rate (%): 43.83561643835616
Overall Max Profit (%): 40.00000000000001
Overall Max Loss (%): -10.000000000000004
Average Return on Profitable Trade (%): 32.60125479103593
Average Loss on Losing Trade (%): -9.108139285191424
Reward to Risk Ratio: 3.579354000881504
Daily returns:0.06297244795001189
Average Holding Period for Winning Trades (days): 224.1796875
Average Holding Period for Losing Trades (days): 84.45731707317073
R1:20184.548535712816
R2:14542.4659810162
R3:12999.691779611203
R4:19227.87731634459
R5:35.850374054967524
#Mid Cap
Overall Average Return (%): 7.792130877188311
Overall Average Holding Period (days): 92.54105571847508
Total Number of Trades: 682
Overall Strike Rate (%): 38.56304985337243
Overall Max Profit (%): 40.00000000000001
Overall Max Loss (%): -10.000000000000004
Average Return on Profitable Trade (%): 35.56800102400423
Average Loss on Losing Trade (%): -9.642365181552952
Reward to Risk Ratio: 3.6887216314986966
Daily returns:0.08420188009193712
Average Holding Period for Winning Trades (days): 147.27376425855513
Average Holding Period for Losing Trades (days): 58.18615751789976
R1:13429.95173522133
R2:24150.941753319166
R3:17823.65414299831
R4:21015.634587671066
R5:39.99368759352376
#Small Cap
Overall Average Return (%): 7.395393777537624
Overall Average Holding Period (days): 74.49244332493703
Total Number of Trades: 794
Overall Strike Rate (%): 36.14609571788413
Overall Max Profit (%): 40.0
Overall Max Loss (%): -10.000000000000004
Average Return on Profitable Trade (%): 37.70763536701336
Average Loss on Losing Trade (%): -9.763606885538387
Reward to Risk Ratio: 3.8620599752807516
Daily returns:0.09927710043391674
Average Holding Period for Winning Trades (days): 117.29268292682927
Average Holding Period for Losing Trades (days): 50.26429980276134
R1:11521.048645383262
R2:32148.327096018875
R3:20376.52294766372
R4:22176.334489674264
R5:47.56322959811979
#Micro Cap
Overall Average Return (%): 6.530381964880467
Overall Average Holding Period (days): 64.84764542936288
Total Number of Trades: 1805
Overall Strike Rate (%): 34.016620498614955
Overall Max Profit (%): 40.00000000000001
Overall Max Loss (%): -10.000000000000004
Average Return on Profitable Trade (%): 38.26100560699482
Average Loss on Losing Trade (%): -9.82780688168394
Reward to Risk Ratio: 3.8931377129827167
Daily returns:0.1007034553319884
Average Holding Period for Winning Trades (days): 102.20032573289902
Average Holding Period for Losing Trades (days): 45.591099916036946
R1:8868.15173846847
R2:37437.263856663354
R3:22318.413318997955
R4:21724.96044560496
R5:49.06459586916915

In [ ]:
Stock('TCS')

[['TCS', Timestamp('2016-05-02 03:45:00'), Timestamp('2016-08-29 03:45:00'), 1264.5, 1267.475, 0.23527085804665157, 119], ['TCS', Timestamp('2017-07-24 03:45:00'), Timestamp('2017-10-03 03:45:00'), 1241.0, 1220.0, -1.6921837228041903, 71], ['TCS', Timestamp('2017-10-16 03:45:00'), Timestamp('2018-12-31 03:45:00'), 1277.525, 1908.0, 49.35128471067101, 441], ['TCS', Timestamp('2019-06-10 03:45:00'), Timestamp('2019-07-08 03:45:00'), 2196.7, 2149.0, -2.17143897664678, 28], ['TCS', Timestamp('2019-08-05 03:45:00'), Timestamp('2019-09-30 03:45:00'), 2199.8999, 1979.9099099999999, -10.000000000000002, 56], ['TCS', Timestamp('2020-09-14 03:45:00'), Timestamp('2021-05-03 03:45:00'), 2384.1001, 3024.8999, 26.8780576788701, 231], ['TCS', Timestamp('2021-08-09 03:45:00'), Timestamp('2022-01-31 03:45:00'), 3323.8999, 3749.0, 12.789196810650047, 175], ['TCS', Timestamp('2023-09-18 03:45:00'), Timestamp('2024-06-03 03:45:00'), 3580.05, 3732.8, 4.266700185751596, 259], ['TCS', Timestamp('2024-08-26 0

([['TCS',
   Timestamp('2016-05-02 03:45:00'),
   Timestamp('2016-08-29 03:45:00'),
   1264.5,
   1267.475,
   0.23527085804665157,
   119],
  ['TCS',
   Timestamp('2017-07-24 03:45:00'),
   Timestamp('2017-10-03 03:45:00'),
   1241.0,
   1220.0,
   -1.6921837228041903,
   71],
  ['TCS',
   Timestamp('2017-10-16 03:45:00'),
   Timestamp('2018-12-31 03:45:00'),
   1277.525,
   1908.0,
   49.35128471067101,
   441],
  ['TCS',
   Timestamp('2019-06-10 03:45:00'),
   Timestamp('2019-07-08 03:45:00'),
   2196.7,
   2149.0,
   -2.17143897664678,
   28],
  ['TCS',
   Timestamp('2019-08-05 03:45:00'),
   Timestamp('2019-09-30 03:45:00'),
   2199.8999,
   1979.9099099999999,
   -10.000000000000002,
   56],
  ['TCS',
   Timestamp('2020-09-14 03:45:00'),
   Timestamp('2021-05-03 03:45:00'),
   2384.1001,
   3024.8999,
   26.8780576788701,
   231],
  ['TCS',
   Timestamp('2021-08-09 03:45:00'),
   Timestamp('2022-01-31 03:45:00'),
   3323.8999,
   3749.0,
   12.789196810650047,
   175],
  ['TCS',


In [ ]:
if universe == 'Small Cap':
    symbols_df = pd.read_csv('/content/ind_niftysmallcap100list.csv')
elif universe == 'Large Cap':
    symbols_df = pd.read_csv('/content/ind_nifty50list.csv')
elif universe == 'Mid Cap':
    symbols_df = pd.read_csv('/content/ind_niftymidcap100list.csv')
elif universe == 'Micro Cap':
    symbols_df = pd.read_csv('/content/ind_niftymicrocap250_list.csv')
elif universe == 'nifty500':
    symbols_df = pd.read_csv('/content/ind_nifty500list.csv')

# Print column names to check
#print("Column names in the CSV:", symbols_df.columns)



if __name__ == '__main__':
     NAME,strategy_summary,performance_summary, daily_returns = main()
#Stock('ACC')

Processing 360 ONE WAM Ltd. (360ONE)
[['360ONE', Timestamp('2020-06-08 03:45:00'), Timestamp('2020-10-19 03:45:00'), np.float64(251.75), np.float64(226.57500000000002), np.float64(-9.999999999999993), 133], ['360ONE', Timestamp('2020-11-17 03:45:00'), Timestamp('2020-12-28 03:45:00'), np.float64(243.337495), np.float64(245.75), np.float64(0.9914234549016009), 41], ['360ONE', Timestamp('2021-04-05 03:45:00'), Timestamp('2021-04-19 03:45:00'), np.float64(327.0), np.float64(294.3), np.float64(-9.999999999999996), 14], ['360ONE', Timestamp('2021-05-17 03:45:00'), Timestamp('2021-05-24 03:45:00'), np.float64(304.25), np.float64(273.825), np.float64(-10.000000000000004), 7], ['360ONE', Timestamp('2021-09-20 03:45:00'), Timestamp('2021-12-06 03:45:00'), np.float64(405.0), np.float64(364.5), np.float64(-10.0), 77], ['360ONE', Timestamp('2022-01-10 03:45:00'), Timestamp('2022-06-20 03:45:00'), np.float64(381.9375), np.float64(343.74375000000003), np.float64(-9.999999999999991), 161], ['360ONE',

ERROR:tvDatafeed.main:Connection timed out
ERROR:tvDatafeed.main:no data, please check the exchange and symbol


No data found for GET_D. Skipping...
Processing GMR Airports Infrastructure Ltd. (GMRINFRA)
[['GMRINFRA', Timestamp('2016-09-12 03:45:00'), Timestamp('2016-10-03 03:45:00'), np.float64(13.06315787), np.float64(11.756842083), np.float64(-9.999999999999993), 21], ['GMRINFRA', Timestamp('2016-11-01 03:45:00'), Timestamp('2016-11-07 03:45:00'), np.float64(12.47554177), np.float64(11.227987593), np.float64(-9.999999999999998), 6], ['GMRINFRA', Timestamp('2017-04-24 03:45:00'), Timestamp('2017-05-08 03:45:00'), np.float64(16.36284826), np.float64(14.726563434000001), np.float64(-9.999999999999993), 14], ['GMRINFRA', Timestamp('2017-06-12 03:45:00'), Timestamp('2017-08-14 03:45:00'), np.float64(15.32322069), np.float64(13.790898621), np.float64(-9.999999999999995), 63], ['GMRINFRA', Timestamp('2017-10-30 03:45:00'), Timestamp('2017-11-20 03:45:00'), np.float64(16.45325165), np.float64(14.807926485), np.float64(-9.999999999999995), 21], ['GMRINFRA', Timestamp('2018-04-09 03:45:00'), Timestamp(

I have given some weights to R1,R2...R5 so all are in 10^4 range

In [ ]:
 scope = ["https://spreadsheets.google.com/feeds", 'https://www.googleapis.com/auth/spreadsheets',
                 "https://www.googleapis.com/auth/drive.file", "https://www.googleapis.com/auth/drive"]

 creds = ServiceAccountCredentials.from_json_keyfile_name('/content/invststrat-2ea1f80133e9.json', scope)
 client = gspread.authorize(creds)

        # Open the Google Sheet
 sheet = client.open_by_url('https://docs.google.com/spreadsheets/d/1bpU1qXdEnfIR4AzZyJ_krR1zjOB-8Ix3cIB8V0iO5bg/edit?usp=sharing')
 worksheet = sheet.get_worksheet(0)  # Select the first worksheet

 existing_data = worksheet.get_all_values()
 next_row = len(existing_data) + 1

    # Append the strategy summary to 'strategy' column, performance summary to 'performance' column, and daily returns to 'daily_returns' column
 worksheet.update_cell(next_row, 1, NAME)
 worksheet.update_cell(next_row, 2, strategy_summary)

 worksheet.update_cell(next_row, 3, performance_summary)
 worksheet.update_cell(next_row, 4, daily_returns)


FileNotFoundError: [Errno 2] No such file or directory: '/content/invststrat-2ea1f80133e9.json'

In [ ]:
df['Daily Return'].iloc[i] = df['close'].iloc[i].pct_change()

# Display the daily returns
print(df[['close', 'Daily Return']].head())

NameError: name 'df' is not defined